In [1]:
!pip install magic-pdf[full] -q
!magic-pdf --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [5]:
!wget "https://huggingface.co/datasets/dengchao/LongDocURL/resolve/main/LongDocURL_public_with_subtask_category.jsonl" -O LongDocURL.jsonl

--2026-03-23 07:33:21--  https://huggingface.co/datasets/dengchao/LongDocURL/resolve/main/LongDocURL_public_with_subtask_category.jsonl
Resolving huggingface.co (huggingface.co)... 3.168.73.111, 3.168.73.38, 3.168.73.106, ...
Connecting to huggingface.co (huggingface.co)|3.168.73.111|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/dengchao/LongDocURL/5c96dbe9055f0382f207228d0f84b77206b6bcb6/LongDocURL_public_with_subtask_category.jsonl?%2Fdatasets%2Fdengchao%2FLongDocURL%2Fresolve%2Fmain%2FLongDocURL_public_with_subtask_category.jsonl=&etag=%22c7e90195e29c7e00d6be977da6dd2d4374003c62%22 [following]
--2026-03-23 07:33:21--  https://huggingface.co/api/resolve-cache/datasets/dengchao/LongDocURL/5c96dbe9055f0382f207228d0f84b77206b6bcb6/LongDocURL_public_with_subtask_category.jsonl?%2Fdatasets%2Fdengchao%2FLongDocURL%2Fresolve%2Fmain%2FLongDocURL_public_with_subtask_category.jsonl=&etag=%22c7e90195e29c7e00d6be977da6dd2d

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import json

# 取 usable_data 里的第一个文档
with open("LongDocURL.jsonl") as f:
    data = [json.loads(line) for line in f]

# 找前5个不同的 doc_no
doc_nos = []
for d in data:
    if d["doc_no"] not in doc_nos:
        doc_nos.append(d["doc_no"])
    if len(doc_nos) == 5:
        break

print("测试用的5个文档:", doc_nos)

# 看看这5个文档对应的 QA 条数
for doc in doc_nos:
    qs = [d for d in data if d["doc_no"] == doc]
    print(f"  {doc}: {len(qs)} 条 QA, evidence_pages 示例: {qs[0]['evidence_pages']}")

测试用的5个文档: ['4026369', '4125651', '4145761', '4091930', '4088207']
  4026369: 2 条 QA, evidence_pages 示例: [67]
  4125651: 2 条 QA, evidence_pages 示例: [5, 9]
  4145761: 9 条 QA, evidence_pages 示例: [111, 112]
  4091930: 12 条 QA, evidence_pages 示例: [47]
  4088207: 25 条 QA, evidence_pages 示例: [20]


In [10]:
import os

# 找 PDF 文件位置
!find /content/drive/MyDrive -name "4088207.pdf" 2>/dev/null

In [11]:
! ls /content/drive/MyDrive/datasets/LongDocURL

data-00000-of-00006.arrow  data-00003-of-00006.arrow  dataset_info.json
data-00001-of-00006.arrow  data-00004-of-00006.arrow  state.json
data-00002-of-00006.arrow  data-00005-of-00006.arrow


In [12]:
from datasets import load_from_disk
import os

# 加载 Arrow 数据集
DRIVE_PATH = "/content/drive/MyDrive/datasets/LongDocURL"  # 改成你的实际路径
ds = load_from_disk(DRIVE_PATH)

print(f"行数: {len(ds)}")
print(f"字段: {ds.column_names}")

# 看一条确认结构
row = ds[0]
for k, v in row.items():
    print(f"  [{k}] {type(v).__name__}: {repr(str(v)[:80])}")

行数: 34309
字段: ['__key__', '__url__', 'pdf']
  [__key__] str: 'mnt/achao/Downloads/ccpdf_zip/4000-4999/4118/4118517'
  [__url__] str: '/root/.cache/huggingface/hub/datasets--dengchao--LongDocURL/snapshots/5c96dbe905'
  [pdf] bytes: "b'%PDF-1.4\\r%\\xe2\\xe3\\xcf\\xd3\\r\\n203 0 obj<</H[596 657]/Linearized 1/E 54648/L 3"


In [13]:
import os

PDF_OUT = "/content/test_pdfs"
os.makedirs(PDF_OUT, exist_ok=True)

# 提取5个测试文档
target_docs = ['4026369', '4125651', '4145761', '4091930', '4088207']

saved = {}
for row in ds:
    # __key__ 格式是 'mnt/.../4088/4088207'，取最后一段
    doc_no = row["__key__"].split("/")[-1]
    if doc_no in target_docs and doc_no not in saved:
        pdf_path = f"{PDF_OUT}/{doc_no}.pdf"
        with open(pdf_path, "wb") as f:
            f.write(row["pdf"])
        saved[doc_no] = pdf_path
        print(f"已保存: {pdf_path}  ({len(row['pdf'])/1024:.0f} KB)")
    if len(saved) == len(target_docs):
        break

print(f"\n共提取 {len(saved)} 个 PDF")

已保存: /content/test_pdfs/4145761.pdf  (11476 KB)
已保存: /content/test_pdfs/4125651.pdf  (4890 KB)
已保存: /content/test_pdfs/4026369.pdf  (35 KB)
已保存: /content/test_pdfs/4091930.pdf  (4348 KB)
已保存: /content/test_pdfs/4088207.pdf  (1755 KB)

共提取 5 个 PDF


In [15]:
import json, os

# 创建 MinerU 配置文件
config = {
    "bucket_info": {},
    "models-dir": "/root/magic-pdf-models",
    "layoutreader-model-dir": "/root/magic-pdf-models/layoutreader",
    "device-mode": "cpu",
    "layout-config": {
        "model": "layoutlmv3"
    },
    "formula-config": {
        "mfd_model": "yolo_v8_mfd",
        "mfr_model": "unimernet_small",
        "enable": False
    },
    "table-config": {
        "model": "rapid_table",
        "enable": False,
        "max_time": 400
    }
}

with open("/root/magic-pdf.json", "w") as f:
    json.dump(config, f, indent=2)

print("配置文件已创建")

# 下载模型权重
!pip install huggingface_hub -q
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="opendatalab/PDF-Extract-Kit-1.0",
    local_dir="/root/magic-pdf-models",
    ignore_patterns=["*.onnx"]   # 跳过 onnx 文件节省空间
)
print("模型下载完成")

配置文件已创建


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 180 files:   0%|          | 0/180 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/835 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

ppocr_keys_v1.txt: 0.00B [00:00, ?B/s]

table_master_structure_dict.txt:   0%|          | 0.00/435 [00:00<?, ?B/s]

models/TabRec/TableMaster/table_structur(…):   0%|          | 0.00/3.39M [00:00<?, ?B/s]

inference.pdiparams.info:   0%|          | 0.00/25.9k [00:00<?, ?B/s]

模型下载完成


In [17]:
import json

config = {
    "bucket_info": {},
    "models-dir": "/root/magic-pdf-models",
    "layoutreader-model-dir": "/root/magic-pdf-models/layoutreader",
    "device-mode": "cpu",
    "layout-config": {
        "model": "doclayout_yolo"
    },
    "formula-config": {
        "mfd_model": "yolo_v8_mfd",
        "mfr_model": "unimernet_small",
        "enable": False
    },
    "table-config": {
        "model": "rapid_table",
        "enable": False,
        "max_time": 400
    }
}

with open("/root/magic-pdf.json", "w") as f:
    json.dump(config, f, indent=2)

print("配置更新完成")

配置更新完成


In [19]:
from huggingface_hub import hf_hub_download

# 下载 doclayout_yolo 模型
hf_hub_download(
    repo_id="opendatalab/PDF-Extract-Kit-1.0",
    filename="models/Layout/YOLO/doclayout_yolo_docstructbench_imgsz1280_2501.pt",
    local_dir="/root/magic-pdf-models",
)
print("下载完成")

# 移动到正确路径
import shutil, os
os.makedirs("/root/magic-pdf-models/Layout/YOLO", exist_ok=True)
src = "/root/magic-pdf-models/models/Layout/YOLO/doclayout_yolo_docstructbench_imgsz1280_2501.pt"
dst = "/root/magic-pdf-models/Layout/YOLO/doclayout_yolo_docstructbench_imgsz1280_2501.pt"
if os.path.exists(src) and not os.path.exists(dst):
    shutil.move(src, dst)
    print("移动完成")

下载完成
移动完成


In [20]:
# 用 MinerU 解析 4088207.pdf
!magic-pdf \
    -p /content/test_pdfs/4088207.pdf \
    -o /content/mineru_output \
    -m auto

# 看输出文件结构
!find /content/mineru_output -type f | head -30

2026-03-23 07:42:40.246 | WARNING  | magic_pdf.libs.config_reader:get_latex_delimiter_config:132 - 'latex-delimiter-config' not found in magic-pdf.json, use 'None' as default
2026-03-23 07:42:45.041 | INFO     | magic_pdf.data.dataset:__init__:157 - lang: None
2026-03-23 07:42:46.934 | INFO     | magic_pdf.libs.pdf_check:detect_invalid_chars:67 - cid_count: 0, text_len: 25799, cid_chars_radio: 0.0
2026-03-23 07:42:50.028 | INFO     | magic_pdf.model.doc_analyze_by_custom_model:doc_analyze:162 - Batch 1/1: 76 pages/76 pages
2026-03-23 07:42:50.030 | INFO     | magic_pdf.model.pdf_extract_kit:__init__:68 - DocAnalysis init, this may take some times, layout_model: doclayout_yolo, apply_formula: False, apply_ocr: True, apply_table: False, table_model: rapid_table, lang: None
2026-03-23 07:42:50.031 | INFO     | magic_pdf.model.pdf_extract_kit:__init__:82 - using device: cpu
2026-03-23 07:42:50.031 | INFO     | magic_pdf.model.pdf_extract_kit:__init__:86 - using models_dir: /root/magic-pdf-

In [21]:
!pip install pymupdf -q

import fitz  # pymupdf
import json

def extract_pages(pdf_path):
    """提取 PDF 每页文本，返回 {page_idx: text} 字典（0-indexed）"""
    doc = fitz.open(pdf_path)
    pages = {}
    for i, page in enumerate(doc):
        pages[i] = page.get_text()
    doc.close()
    return pages

# 测试 4088207
pages = extract_pages("/content/test_pdfs/4088207.pdf")
print(f"总页数: {len(pages)}")
print(f"\n--- 第20页（evidence page 示例）---")
print(pages.get(19, "")[:500])  # 0-indexed，第20页是index 19

总页数: 76

--- 第20页（evidence page 示例）---
Nordic Fibreboard AS Annual Report (audited) 
2020 
20 
CONSOLIDATED FINANCIAL STATEMENTS 
CONSOLIDATED STATEMENT OF FINANCIAL POSITION
€ thousand 
31.12.2020 
31.12.2019 
Cash and cash equivalents 
26 
7 
Receivables and prepayments (Note 5) 
794 
1,394 
Inventories (Note 6) 
544 
894 
Total current assets 
1,364 
2,296 
Investment property (Note 7) 
1,134 
1,121 
Financial assets at fair value through profit or loss (Note 9) 
451 
397 
Property, plant and equipment (Note 8) 
4,695 
5,212 
Inta


In [23]:
import re, string, json, fitz

def normalize(text: str) -> str:
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()

# 重新加载数据
with open("LongDocURL.jsonl") as f:
    data = [json.loads(line) for line in f]

print(f"数据加载完成: {len(data)} 条")

数据加载完成: 2325 条


In [24]:
# 取 4088207 的所有 QA，在对应页面文本里搜 answer
qa_4088207 = [d for d in data if d["doc_no"] == "4088207"]
print(f"4088207 共 {len(qa_4088207)} 条 QA\n")

hit, miss = 0, 0
for d in qa_4088207:
    answer        = str(d["answer"])
    evidence_pgs  = d["evidence_pages"]  # 1-indexed

    # 拼接所有 evidence page 的文本
    ev_text = " ".join(pages.get(pg - 1, "") for pg in evidence_pgs)

    # 匹配
    strict = answer.lower().strip() in ev_text.lower()
    loose  = normalize(answer) in normalize(ev_text)
    norm_a = re.sub(r'[\s,]', '', answer)
    norm_e = re.sub(r'[\s,]', '', ev_text)
    number = norm_a in norm_e

    matched = strict or loose or number
    status  = "✅" if matched else "❌"

    if matched: hit += 1
    else: miss += 1

    print(f"{status} Q: {d['question'][:60]}")
    print(f"   A: {repr(answer[:50])}  pages={evidence_pgs}")
    if not matched:
        print(f"   evidence snippet: {repr(ev_text[:100])}")
    print()

print(f"命中: {hit}/{len(qa_4088207)}  ({hit/len(qa_4088207)*100:.1f}%)")

4088207 共 25 条 QA

✅ Q: What is the total amount of liabilities for Nordic Fibreboar
   A: '5002'  pages=[20]

❌ Q: Where can we find information ensuring the accuracy and comp
   A: '["Independent auditor\'s report", \'Report on other '  pages=[68, 73]
   evidence snippet: ' \nTranslation note: \nThis version of our report is a translation from the original, which was prepar'

❌ Q: Where can we find detailed financial information and account
   A: "['CONSOLIDATED FINANCIAL STATEMENTS', 'NOTES TO TH"  pages=[20, 24]
   evidence snippet: 'Nordic Fibreboard AS Annual Report (audited) \n2020 \n20 \nCONSOLIDATED FINANCIAL STATEMENTS \nCONSOLIDA'

✅ Q: What percentage of the share capital is held by institutiona
   A: '79'  pages=[13]

❌ Q: What is absolute difference value between the total water us
   A: '36.1'  pages=[18]
   evidence snippet: 'Nordic Fibreboard AS Annual Report (audited) \n2020 \n18 \nensures that all end consumers may return th'

❌ Q: Which sections discuss the operatio

In [25]:
import ast

def match_answer(answer, page_text):
    answer = str(answer)

    # 严格匹配
    if answer.lower().strip() in page_text.lower():
        return True, "strict"

    # 宽松匹配
    if normalize(answer) in normalize(page_text):
        return True, "loose"

    # 数字归一化
    norm_a = re.sub(r'[\s,]', '', answer.strip('"\''))
    norm_p = re.sub(r'[\s,]', '', page_text)
    if norm_a and len(norm_a) > 1 and norm_a in norm_p:
        return True, "number"

    # List 拆分：每个元素单独匹配
    try:
        items = ast.literal_eval(answer)
        if isinstance(items, list):
            items = [str(x).strip().strip('"\'') for x in items if len(str(x).strip()) > 2]
            if items and any(normalize(item) in normalize(page_text) for item in items):
                return True, "list_any"
    except:
        pass

    # 去引号再试
    clean = answer.strip('"\'')
    if len(clean) > 2 and normalize(clean) in normalize(page_text):
        return True, "strip_quotes"

    return False, None

# 重新跑 4088207
doc_no = "4088207"
doc_qs = [d for d in data if d["doc_no"] == doc_no]
pages  = extract_pages(f"/content/test_pdfs/{doc_no}.pdf")

hits = 0
methods = []
for d in doc_qs:
    answer = str(d["answer"])
    ev_pages = [p - 1 for p in d["evidence_pages"]]  # 0-indexed
    ev_text  = " ".join(pages.get(p, "") for p in ev_pages)

    matched, method = match_answer(answer, ev_text)
    if matched:
        hits += 1
        methods.append(method)
        print(f"✅ [{method}] A: {answer[:50]}")
    else:
        print(f"❌ A: {answer[:50]}")
        print(f"   page snippet: {ev_text[:100]}")

from collections import Counter
print(f"\n命中: {hits}/{len(doc_qs)}  ({hits/len(doc_qs)*100:.1f}%)")
print(f"方法分布: {Counter(methods)}")

✅ [loose] A: 5002
✅ [list_any] A: ["Independent auditor's report", 'Report on other 
✅ [list_any] A: ['CONSOLIDATED FINANCIAL STATEMENTS', 'NOTES TO TH
✅ [strict] A: 79
❌ A: 36.1
   page snippet: Nordic Fibreboard AS Annual Report (audited) 
2020 
18 
ensures that all end consumers may return th
✅ [list_any] A: ['PERFORMANCE OF BUSINESS UNITS', 'SKANO FURNITURE
✅ [list_any] A: ['SHARE CAPITAL BY THE TYPE OF THE OWNERS AS OF 31
✅ [loose] A: ['RISKS']
✅ [loose] A: ['FORECAST AND DEVELOPMENT', 'BUSINESS ENVIRONMENT
✅ [strict] A: 26
❌ A: ['FINACIAL RATIOS', 'CONSOLIDATED STATEMENT OF FIN
   page snippet: Nordic Fibreboard AS Annual Report (audited) 
2020 
11 
FINANCIAL RATIOS 
€ thousand 
Income stateme
❌ A: 62.2
   page snippet: Nordic Fibreboard AS Annual Report (audited) 
2020 
18 
ensures that all end consumers may return th
❌ A: 3250804
   page snippet: Nordic Fibreboard AS Annual Report (audited) 
2020 
13 
SHARE CAPITAL GEOGRAPHICALLY AS OF 31.12.202
✅ [strict] A: production and dis

In [28]:
import fitz, ast, re, string, json
from collections import Counter

def extract_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = {i: page.get_text() for i, page in enumerate(doc)}
    doc.close()
    return pages

# 从 Arrow 数据集里提取所有 PDF
from datasets import load_from_disk
ds = load_from_disk("/content/drive/MyDrive/datasets/LongDocURL")  # 改成你的路径

# 建立 doc_no → pdf bytes 的映射
print("建立 PDF 索引...")
pdf_map = {}
for row in ds:
    doc_no = row["__key__"].split("/")[-1]
    pdf_map[doc_no] = row["pdf"]
print(f"共 {len(pdf_map)} 个文档")

# 全量匹配
results = []
doc_cache = {}  # 缓存已解析的页面文本

for i, d in enumerate(data):
    doc_no  = d["doc_no"]
    answer  = str(d["answer"])
    ev_pages = [p - 1 for p in d["evidence_pages"]]

    # 解析 PDF（缓存）
    if doc_no not in doc_cache:
        if doc_no in pdf_map:
            import tempfile, os
            with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as f:
                f.write(pdf_map[doc_no])
                tmp = f.name
            doc_cache[doc_no] = extract_pages(tmp)
            os.unlink(tmp)
        else:
            doc_cache[doc_no] = {}

    pages = doc_cache[doc_no]
    ev_text = " ".join(pages.get(p, "") for p in ev_pages)

    matched, method = match_answer_v2(answer, ev_text)
    results.append({
        "question_id":      d["question_id"],
        "doc_no":           doc_no,
        "question":         d["question"],
        "answer":           answer,
        "answer_format":    d["answer_format"],
        "task_tag":         d["task_tag"],
        "evidence_pages":   d["evidence_pages"],
        "evidence_sources": d["evidence_sources"],
        "detailed_evidences": str(d.get("detailed_evidences", "")),
        "matched":          matched,
        "match_method":     method,
    })

    if (i+1) % 100 == 0:
        matched_so_far = sum(r["matched"] for r in results)
        print(f"  {i+1}/{len(data)}  命中率: {matched_so_far/(i+1)*100:.1f}%")

# 汇总
matched_total = sum(r["matched"] for r in results)
print(f"\n=== 全量结果 ===")
print(f"总样本:   {len(results)}")
print(f"可用样本: {matched_total}  ({matched_total/len(results)*100:.1f}%)")
print(f"方法分布: {Counter(r['match_method'] for r in results if r['matched'])}")

# 保存
import json
with open("/content/drive/MyDrive/datasets/longdocurl_matched_full.jsonl", "w") as f:
    for r in results:
        if r["matched"]:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("已保存 longdocurl_matched_full.jsonl")

建立 PDF 索引...
共 34309 个文档
  100/2325  命中率: 62.0%
  200/2325  命中率: 65.5%
  300/2325  命中率: 67.3%
  400/2325  命中率: 68.8%
  500/2325  命中率: 67.6%
  600/2325  命中率: 68.3%
  700/2325  命中率: 68.3%
  800/2325  命中率: 67.9%
  900/2325  命中率: 68.8%
  1000/2325  命中率: 69.1%
  1100/2325  命中率: 68.5%
  1200/2325  命中率: 68.8%
  1300/2325  命中率: 68.7%
  1400/2325  命中率: 68.6%
  1500/2325  命中率: 68.2%
  1600/2325  命中率: 68.4%
  1700/2325  命中率: 68.6%
  1800/2325  命中率: 68.5%
  1900/2325  命中率: 68.5%
  2000/2325  命中率: 68.3%
  2100/2325  命中率: 68.1%
  2200/2325  命中率: 68.3%
  2300/2325  命中率: 68.3%

=== 全量结果 ===
总样本:   2325
可用样本: 1586  (68.2%)
方法分布: Counter({'strict': 773, 'list_any': 451, 'loose': 322, 'number': 40})
已保存 longdocurl_matched_full.jsonl


In [29]:
import fitz, re

def chunk_page_text(text, max_tokens=512, overlap_tokens=64, min_chunk_tokens=30):
    """
    把单页文本切成 chunk：
    1. 先按 \n\n 切段落
    2. 合并短段落直到接近 max_tokens
    3. 超过就截断，保留 overlap
    """
    def token_len(t): return len(t.split())

    # 按段落切
    paras = [p.strip() for p in re.split(r'\n{2,}', text) if len(p.strip().split()) >= 5]

    chunks = []
    current = []
    current_len = 0

    for para in paras:
        para_len = token_len(para)

        if current_len + para_len > max_tokens and current:
            chunk_text = " ".join(current)
            chunks.append(chunk_text)
            # overlap：保留最后几个 token
            overlap_text = " ".join(chunk_text.split()[-overlap_tokens:])
            current = [overlap_text]
            current_len = token_len(overlap_text)

        current.append(para)
        current_len += para_len

    if current and current_len >= min_chunk_tokens:
        chunks.append(" ".join(current))

    return chunks

# 测试 4088207，重点看 evidence page 20（有表格）和 page 13
doc = fitz.open("/content/test_pdfs/4088207.pdf")

for page_idx in [12, 19, 22]:  # 0-indexed: 第13、20、23页
    text = doc[page_idx].get_text()
    chunks = chunk_page_text(text)
    print(f"\n{'='*60}")
    print(f"第 {page_idx+1} 页  →  {len(chunks)} 个 chunk")
    for i, c in enumerate(chunks):
        print(f"\n  [chunk {i}] ({len(c.split())} tokens)")
        print(f"  {c[:200]}...")

doc.close()


第 13 页  →  1 个 chunk

  [chunk 0] (278 tokens)
  Nordic Fibreboard AS Annual Report (audited) 
2020 
13 
SHARE CAPITAL GEOGRAPHICALLY AS OF 31.12.2020: 
Number of 
shareholders 
% from 
shareholders 
Number of shares 
% from share 
capital 
Estonia ...

第 20 页  →  1 个 chunk

  [chunk 0] (187 tokens)
  Nordic Fibreboard AS Annual Report (audited) 
2020 
20 
CONSOLIDATED FINANCIAL STATEMENTS 
CONSOLIDATED STATEMENT OF FINANCIAL POSITION
€ thousand 
31.12.2020 
31.12.2019 
Cash and cash equivalents 
2...

第 23 页  →  1 个 chunk

  [chunk 0] (190 tokens)
  Nordic Fibreboard AS Annual Report (audited) 
2020 
23 
CONSOLIDATED STATEMENT OF CHANGES IN EQUITY 
€ thousand 
Share 
capital 
Share 
premium 
Statutory 
reserve 
capital 
Other 
reserves 
Retained ...


In [33]:
import fitz

doc = fitz.open("/content/test_pdfs/4088207.pdf")
page = doc[19]  # 第20页

# 看文本块结构
print("=== 文本块 (get_text('blocks')) ===")
blocks = page.get_text("blocks")
for b in blocks[:10]:
    # (x0, y0, x1, y1, text, block_no, block_type)
    # block_type: 0=text, 1=image
    btype = "TEXT" if b[6] == 0 else "IMAGE"
    print(f"[{btype}] block#{b[5]}  '{str(b[4])[:80].strip()}'")

print("\n=== 图片 ===")
imgs = page.get_images()
print(f"图片数量: {len(imgs)}")
for img in imgs:
    print(f"  xref={img[0]}  {img[1]}x{img[2] if len(img)>2 else '?'}")

print("\n=== 表格 ===")
tabs = page.find_tables()
print(f"表格数量: {len(tabs.tables)}")
for i, t in enumerate(tabs.tables):
    print(f"  table#{i}  rows={t.row_count}  cols={t.col_count}  bbox={t.bbox}")

doc.close()

=== 文本块 (get_text('blocks')) ===
[TEXT] block#0  'Nordic Fibreboard AS Annual Report (audited) 
2020'
[TEXT] block#1  '20'
[TEXT] block#2  'CONSOLIDATED FINANCIAL STATEMENTS'
[TEXT] block#3  'CONSOLIDATED STATEMENT OF FINANCIAL POSITION'
[TEXT] block#4  '€ thousand 
31.12.2020 
31.12.2019'
[TEXT] block#5  'Cash and cash equivalents 
26 
7'
[TEXT] block#6  'Receivables and prepayments (Note 5) 
794 
1,394'
[TEXT] block#7  'Inventories (Note 6) 
544 
894'
[TEXT] block#8  'Total current assets 
1,364 
2,296'
[TEXT] block#9  'Investment property (Note 7) 
1,134 
1,121'

=== 图片 ===
图片数量: 2
  xref=254  251x1159
  xref=260  256x685

=== 表格 ===
表格数量: 1
  table#0  rows=32  cols=5  bbox=(83.9279998779297, 144.5999755859375, 538.6281094937711, 569.4800008138021)


In [34]:
import fitz
import re
from typing import List, Dict

def token_len(text: str) -> int:
    return len(text.split())

def merge_text_blocks(blocks, max_tokens=1200, overlap_tokens=100) -> List[Dict]:
    """
    把文本块合并成 text chunk，最大 1200 tokens，overlap 100 tokens
    跳过表格区域内的文本块（避免重复）
    """
    chunks = []
    current_texts = []
    current_len = 0

    for b in blocks:
        if b[6] != 0:  # 跳过非文本块
            continue
        text = b[4].strip()
        if not text or token_len(text) < 2:
            continue

        tl = token_len(text)

        if current_len + tl > max_tokens and current_texts:
            chunk_text = " ".join(current_texts)
            chunks.append({
                "type": "text",
                "text": chunk_text,
                "token_len": token_len(chunk_text),
            })
            # overlap：保留最后 overlap_tokens 个 token
            overlap = " ".join(chunk_text.split()[-overlap_tokens:])
            current_texts = [overlap]
            current_len = token_len(overlap)

        current_texts.append(text)
        current_len += tl

    if current_texts and current_len >= 10:
        chunk_text = " ".join(current_texts)
        chunks.append({
            "type": "text",
            "text": chunk_text,
            "token_len": token_len(chunk_text),
        })

    return chunks


def extract_table_as_chunk(table) -> Dict:
    """把表格提取为文本表示的 chunk"""
    rows = []
    try:
        for row in table.extract():
            clean = [str(c).strip() if c else "" for c in row]
            rows.append(" | ".join(clean))
        text = "\n".join(rows)
    except:
        text = f"[Table bbox={table.bbox}]"
    return {
        "type": "table",
        "text": text,
        "token_len": token_len(text),
        "bbox": list(table.bbox),
    }


def extract_image_as_chunk(page, xref, bbox) -> Dict:
    """图片 chunk：提取图片附近的 caption 文本"""
    # 在图片 bbox 下方 50pt 范围内找 caption
    x0, y0, x1, y1 = bbox
    caption_rect = fitz.Rect(x0, y1, x1, y1 + 50)
    caption = page.get_text("text", clip=caption_rect).strip()
    return {
        "type": "image",
        "text": caption if caption else "[Image]",
        "token_len": token_len(caption) if caption else 0,
        "bbox": list(bbox),
        "xref": xref,
    }


def chunk_pdf_page(page) -> List[Dict]:
    """
    对单页执行完整的 chunk 提取：
    - text chunk：合并段落，最大 1200 tokens，overlap 100
    - table chunk：每个表格独立一个 chunk
    - image chunk：每张图片独立一个 chunk
    """
    chunks = []

    # 1. 找出表格区域（bbox），后续文本块里跳过这些区域
    tables = page.find_tables()
    table_bboxes = [fitz.Rect(t.bbox) for t in tables.tables]

    # 2. 提取文本块，跳过在表格区域内的
    all_blocks = page.get_text("blocks")
    text_blocks = []
    for b in all_blocks:
        if b[6] != 0:
            continue
        block_rect = fitz.Rect(b[:4])
        in_table = any(block_rect.intersects(tb) for tb in table_bboxes)
        if not in_table:
            text_blocks.append(b)

    # 3. 合并文本块成 text chunk
    text_chunks = merge_text_blocks(text_blocks)
    chunks.extend(text_chunks)

    # 4. 每个表格独立一个 chunk
    for t in tables.tables:
        chunks.append(extract_table_as_chunk(t))

    # 5. 每张图片独立一个 chunk
    for img in page.get_images():
        xref = img[0]
        rects = page.get_image_rects(xref)
        if rects:
            bbox = rects[0]
            chunks.append(extract_image_as_chunk(page, xref, bbox))

    return chunks


# ── 测试 4088207 的第13、20、23页 ──────────────────────────
doc = fitz.open("/content/test_pdfs/4088207.pdf")

for page_idx in [12, 19, 22]:
    page = doc[page_idx]
    chunks = chunk_pdf_page(page)
    print(f"\n{'='*60}")
    print(f"第 {page_idx+1} 页  →  {len(chunks)} 个 chunk")
    for i, c in enumerate(chunks):
        print(f"\n  [{c['type']:6s}] chunk#{i}  ({c['token_len']} tokens)")
        print(f"  {c['text'][:150]}")

doc.close()


第 13 页  →  5 个 chunk

  [text  ] chunk#0  (121 tokens)
  Nordic Fibreboard AS Annual Report (audited) 
2020 SHARE CAPITAL GEOGRAPHICALLY AS OF 31.12.2020: SHARE CAPITAL BY THE TYPE OF THE OWNERS AS OF 31.12.

  [table ] chunk#1  (80 tokens)
   | Number of
shareholders | % from
shareholders | Number of shares | % from share
capital
Estonia | 454 | 94% | 4,258,815 | 95%
Finland | 9 | 2% | 38,

  [table ] chunk#2  (46 tokens)
   | Number of
shareholders | % from
shareholders | Number of shares | % from share
capital
Private individuals | 429 | 88% | 936,256 | 21%
Institutiona

  [table ] chunk#3  (112 tokens)
  Shareholder | Number of shares | Shareholding %
PÄRNU HOLDINGS OÜ | 2,682,192 | 60%
Gamma Holding Investment OÜ | 374,968 | 8%
Live Nature OÜ | 100,00

  [image ] chunk#4  (0 tokens)
  [Image]

第 20 页  →  4 个 chunk

  [text  ] chunk#0  (36 tokens)
  Nordic Fibreboard AS Annual Report (audited) 
2020 CONSOLIDATED FINANCIAL STATEMENTS CONSOLIDATED STATEMENT OF FINANCIAL POSITION *Th

In [36]:
def is_header_footer(block_rect, page_height, header_margin=50, footer_margin=50):
    """判断文本块是否在页眉/页脚区域"""
    return block_rect.y0 < header_margin or block_rect.y1 > page_height - footer_margin

def is_decorative_image(pix, min_width=100, min_height=100):
    """过滤装饰性图片：太窄或太矮的通常是背景装饰"""
    return pix.width < min_width or pix.height < min_height

def chunk_pdf_page_v2(page, max_tokens=1200, overlap_tokens=100) -> List[Dict]:
    page_height = page.rect.height
    chunks = []

    # 1. 找表格区域
    tables = page.find_tables()
    table_bboxes = [fitz.Rect(t.bbox) for t in tables.tables]

    # 2. 过滤文本块：跳过页眉页脚 + 表格区域内
    all_blocks = page.get_text("blocks")
    text_blocks = []
    for b in all_blocks:
        if b[6] != 0:
            continue
        block_rect = fitz.Rect(b[:4])
        if is_header_footer(block_rect, page_height):
            continue
        if any(block_rect.intersects(tb) for tb in table_bboxes):
            continue
        text_blocks.append(b)

    # 3. 合并文本块
    text_chunks = merge_text_blocks(text_blocks, max_tokens, overlap_tokens)
    chunks.extend(text_chunks)

    # 4. 表格 chunk
    for t in tables.tables:
        chunks.append(extract_table_as_chunk(t))

    # 5. 图片 chunk：过滤装饰性图片
    doc = page.parent
    for img in page.get_images():
        xref = img[0]
        try:
            pix = fitz.Pixmap(doc, xref)
            if is_decorative_image(pix):
                pix = None
                continue
            pix = None
        except:
            continue

        rects = page.get_image_rects(xref)
        if not rects:
            continue

        bbox = rects[0]
        # 过滤页脚图片
        if bbox.y0 > page_height - 100:
            continue

        # 找 caption：图片上方或下方 60pt
        above = page.get_text("text", clip=fitz.Rect(bbox.x0, bbox.y0-60, bbox.x1, bbox.y0)).strip()
        below = page.get_text("text", clip=fitz.Rect(bbox.x0, bbox.y1, bbox.x1, bbox.y1+60)).strip()
        caption = above or below

        chunks.append({
            "type": "image",
            "text": caption if caption else "[Image]",
            "token_len": token_len(caption) if caption else 0,
            "bbox": list(bbox),
            "xref": xref,
        })

    return chunks

# 测试
doc = fitz.open("/content/test_pdfs/4088207.pdf")

for page_idx in [12, 19, 22]:
    page = doc[page_idx]
    chunks = chunk_pdf_page_v2(page)
    print(f"\n{'='*60}")
    print(f"第 {page_idx+1} 页  →  {len(chunks)} 个 chunk")
    for i, c in enumerate(chunks):
        print(f"\n  [{c['type']:6s}] chunk#{i}  ({c['token_len']} tokens)")
        print(f"  {c['text'][:150]}")

doc.close()


第 13 页  →  5 个 chunk

  [text  ] chunk#0  (114 tokens)
  SHARE CAPITAL GEOGRAPHICALLY AS OF 31.12.2020: SHARE CAPITAL BY THE TYPE OF THE OWNERS AS OF 31.12.2020: LIST OF THE SHAREHOLDERS WITH THE OWNERSHIP M

  [table ] chunk#1  (80 tokens)
   | Number of
shareholders | % from
shareholders | Number of shares | % from share
capital
Estonia | 454 | 94% | 4,258,815 | 95%
Finland | 9 | 2% | 38,

  [table ] chunk#2  (46 tokens)
   | Number of
shareholders | % from
shareholders | Number of shares | % from share
capital
Private individuals | 429 | 88% | 936,256 | 21%
Institutiona

  [table ] chunk#3  (112 tokens)
  Shareholder | Number of shares | Shareholding %
PÄRNU HOLDINGS OÜ | 2,682,192 | 60%
Gamma Holding Investment OÜ | 374,968 | 8%
Live Nature OÜ | 100,00

  [image ] chunk#4  (33 tokens)
  Both Joakim Johan Helenius and Torfinn Losvik have indirect ownership through parent company
Pärnu Holdings OÜ. Indirectly Torfinn Losvik owns shares 

第 20 页  →  3 个 chunk

  [text  ] chunk#0  (29

In [37]:
import json

def chunk_pdf_doc(pdf_path) -> Dict[int, List[Dict]]:
    """处理整个 PDF，返回 {page_idx: [chunks]} （0-indexed）"""
    doc = fitz.open(pdf_path)
    result = {}
    for i in range(len(doc)):
        result[i] = chunk_pdf_page_v2(doc[i])
    doc.close()
    return result

# 处理 4088207
doc_chunks = chunk_pdf_doc("/content/test_pdfs/4088207.pdf")
print(f"总页数: {len(doc_chunks)}")
print(f"总 chunk 数: {sum(len(v) for v in doc_chunks.values())}")
print(f"平均每页: {sum(len(v) for v in doc_chunks.values())/len(doc_chunks):.1f} 个 chunk")

# 验证 pos_chunk 定位：对这个文档的所有 QA 找 pos_chunk
doc_qs = [d for d in data if d["doc_no"] == "4088207"]
print(f"\n该文档 QA 数: {len(doc_qs)}")
print(f"\n{'='*60}")

hit, miss = 0, 0
for d in doc_qs:
    answer   = str(d["answer"])
    ev_pages = [p - 1 for p in d["evidence_pages"]]  # 0-indexed

    pos_chunk = None
    for pg in ev_pages:
        for chunk in doc_chunks.get(pg, []):
            matched, method = match_answer_v2(answer, chunk["text"])
            if matched:
                pos_chunk = chunk
                break
        if pos_chunk:
            break

    if pos_chunk:
        hit += 1
        print(f"✅ A: {answer[:50]}")
        print(f"   chunk type={pos_chunk['type']}  tokens={pos_chunk['token_len']}")
        print(f"   chunk: {pos_chunk['text'][:100]}")
    else:
        miss += 1
        print(f"❌ A: {answer[:50]}")
    print()

print(f"pos_chunk 定位成功: {hit}/{len(doc_qs)}  ({hit/len(doc_qs)*100:.1f}%)")

总页数: 76
总 chunk 数: 246
平均每页: 3.2 个 chunk

该文档 QA 数: 25

✅ A: 5002
   chunk type=table  tokens=278
   chunk: € thousand |  | 31.12.2020 |  | 31.12.2019
Cash and cash equivalents |  | 26 |  | 7
Receivables and 

✅ A: ["Independent auditor's report", 'Report on other 
   chunk type=text  tokens=322
   chunk: Translation note: We also provide those charged with governance with a statement that we have compli

✅ A: ['CONSOLIDATED FINANCIAL STATEMENTS', 'NOTES TO TH
   chunk type=text  tokens=29
   chunk: CONSOLIDATED FINANCIAL STATEMENTS CONSOLIDATED STATEMENT OF FINANCIAL POSITION *The notes to the fin

✅ A: 79
   chunk type=table  tokens=46
   chunk:  | Number of
shareholders | % from
shareholders | Number of shares | % from share
capital
Private in

❌ A: 36.1

✅ A: ['PERFORMANCE OF BUSINESS UNITS', 'SKANO FURNITURE
   chunk type=text  tokens=429
   chunk: to € 1.1 million, property, plant, equipment and intangibles were € 4.7 million as of 31.12.2020 (€ 

✅ A: ['SHARE CAPITAL BY THE TYPE

In [ ]:
import tempfile, os, json
from collections import defaultdict

def get_hard_neg(doc_chunks, pos_chunk, ev_pages, answer):
    """
    hard_neg 策略：同一 evidence page 内，
    类型相同但不包含 answer 的其他 chunk
    """
    candidates = []
    for pg in ev_pages:
        for chunk in doc_chunks.get(pg, []):
            if chunk is pos_chunk:
                continue
            # 同类型
            if chunk["type"] != pos_chunk["type"]:
                continue
            # 不含 answer
            matched, _ = match_answer_v2(answer, chunk["text"])
            if not matched and chunk["token_len"] > 10:
                candidates.append(chunk)

    # 若同页同类型找不到，放宽到同页任意类型
    if not candidates:
        for pg in ev_pages:
            for chunk in doc_chunks.get(pg, []):
                if chunk is pos_chunk:
                    continue
                matched, _ = match_answer_v2(answer, chunk["text"])
                if not matched and chunk["token_len"] > 10:
                    candidates.append(chunk)

    return candidates[0] if candidates else None


# ── 全量处理 ──────────────────────────────────────────────
from datasets import load_from_disk

print("加载 Arrow 数据集...")
ds = load_from_disk("/content/drive/MyDrive/datasets/LongDocURL")
pdf_map = {row["__key__"].split("/")[-1]: row["pdf"] for row in ds}
print(f"PDF 总数: {len(pdf_map)}")

# 加载已匹配的 1586 条
with open("/content/drive/MyDrive/datasets/longdocurl_matched_full.jsonl") as f:
    matched_data = [json.loads(l) for l in f]

print(f"待处理 QA: {len(matched_data)}")

triplets = []
doc_cache = {}
no_pos, no_neg = 0, 0

for i, d in enumerate(matched_data):
    doc_no   = d["doc_no"]
    answer   = d["answer"]
    ev_pages = [p - 1 for p in d["evidence_pages"]]

    # 解析 PDF（缓存）
    if doc_no not in doc_cache:
        if doc_no in pdf_map:
            with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as f_tmp:
                f_tmp.write(pdf_map[doc_no])
                tmp = f_tmp.name
            doc_cache[doc_no] = chunk_pdf_doc(tmp)
            os.unlink(tmp)
        else:
            doc_cache[doc_no] = {}

    doc_chunks = doc_cache[doc_no]

    # 找 pos_chunk
    pos_chunk = None
    for pg in ev_pages:
        for chunk in doc_chunks.get(pg, []):
            matched, method = match_answer_v2(answer, chunk["text"])
            if matched:
                pos_chunk = chunk
                break
        if pos_chunk:
            break

    if not pos_chunk:
        no_pos += 1
        continue

    # 找 hard_neg
    hard_neg = get_hard_neg(doc_chunks, pos_chunk, ev_pages, answer)
    if not hard_neg:
        no_neg += 1
        continue

    triplets.append({
        "question_id":      d["question_id"],
        "doc_no":           doc_no,
        "question":         d["question"],
        "answer":           answer,
        "answer_format":    d["answer_format"],
        "task_tag":         d["task_tag"],
        "evidence_pages":   d["evidence_pages"],
        "pos_chunk":        {"type": pos_chunk["type"], "text": pos_chunk["text"]},
        "hard_neg":         {"type": hard_neg["type"],  "text": hard_neg["text"]},
    })

    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(matched_data)}  三元组: {len(triplets)}  无pos: {no_pos}  无neg: {no_neg}")

print(f"\n=== 最终结果 ===")
print(f"三元组总数:     {len(triplets)}")
print(f"pos_chunk 缺失: {no_pos}")
print(f"hard_neg 缺失:  {no_neg}")

# 保存
out = "/content/drive/MyDrive/datasets/longdocurl_triplets.jsonl"
with open(out, "w") as f:
    for t in triplets:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")
print(f"已保存: {out}")

加载 Arrow 数据集...
PDF 总数: 34309
待处理 QA: 1586
  100/1586  三元组: 40  无pos: 9  无neg: 51
